# Rate-Distortion Curves — Ablation Study

Compares ResSHyp variants (activation × output_padding) and all four architectures.
Data is loaded from a pre-built CSV (`SAR_DDC_FPGA_all_runs_WandB.csv`).
Run `fetch_wandb_runs.py` to regenerate the CSV from W&B.

## Dataset & Run Structure

6 seeds × 10 λ values (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000) = 60 runs per configuration.
Runs are selected by **learning-rate value** (`PROD_LR` per architecture; the ablation pins the old
5e-5 default), never by tag.

| # | Architecture | Activation | Output padding | Notes |
|---|---|---|---|---|
| 1 | ResSHyp | GDN  | ✅ with | Ablation: output_padding effect |
| 2 | ResSHyp | GDN  | ❌ without | Ablation: output_padding effect |
| 3 | ResSHyp | ReLU | ✅ with | Ablation: activation + output_padding |
| 4 | ResSHyp | ReLU | ❌ without | **FPGA baseline** |
| 5 | SHyp    | ReLU | ❌ without | No residual blocks |
| 6 | ResFP   | ReLU | ❌ without | No hyperprior |
| 7 | FP      | ReLU | ❌ without | No residual blocks, no hyperprior |

**`model_statistics` key format:** `{architecture}_{activation}` (e.g. `ResSHyp_relu`) + `_out_pad` suffix when `no_output_padding=False`.


## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

# Quick ANSI colour shortcuts
r = "\033[31m"
y = "\033[33m"
g = "\033[32m"
b = "\033[34m"
e = "\033[0m"

ROOT_DIR = Path("..").resolve()
WANDB_CSV = ROOT_DIR / "notebooks" / "SAR_DDC_FPGA_all_runs_WandB.csv"

# ── Shared color palette (Okabe-Ito, colorblind-safe) ────────────────────
import sys as _sys

_sys.path.insert(0, str(ROOT_DIR / "notebooks"))
del _sys
from _plotkit import PALETTE, export_manuscript, compare_at_lambda
import _plotkit

_plotkit.EXPORT_SUFFIX = ""  # "_new"

SAVE_FIGURES = True
FIGURE_FORMAT = "pdf"  # "pdf"  # "png"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "RD-curves"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Per-architecture production learning rate (selected by the lr-sweep) ──
# Runs are selected by the lr VALUE (CSV `model.net_optimizer.lr` → `lr` column),
# never by tag (most runs carry no lr tag). Most models were retrained:
# SH 5e-5→5e-4, ResSH 5e-5→1e-4; FP/ResFP 5e-5→5e-4.
PROD_LR = {"FP": 5e-4, "ResFP": 5e-4, "SHyp": 5e-4, "ResSHyp": 1e-4}
# The ablation (ReLU vs GDN) stays at the old default — FPGA-incompatible variants (GDN, output_pad) only exist there.
ABLATION_LR = 5e-5

# ── Reference metric set: 500-image MERLIN subset + rANS bitstream bpp ────
METRIC_PREFIX = "test_sub500"
BPP_METRIC = "bpp_bitstream"

# ── Finalized manuscript figures (PDF) land here ──
MANUSCRIPT_DIR = ROOT_DIR / "LaTeX" / "SAR_DDC_FPGA_TGRS_2026" / "figures" / "images"

## Load & Filter Runs

In [ ]:
VERBOSE_FILTERING = True
print(f"{b}Loading and filtering runs... (VERBOSE={VERBOSE_FILTERING}){e}")

# ── Load raw runs from CSV (generated by fetch_wandb_runs.py) ────────────
raw_runs_df = pd.read_csv(WANDB_CSV, index_col="id")
raw_runs_df["tags"] = raw_runs_df["tags"].fillna("[]").apply(json.loads)
raw_runs_df["no_output_padding"] = raw_runs_df["no_output_padding"].map(
    {"True": True, "False": False, True: True, False: False}
)
nb_all_runs = len(raw_runs_df)
print(f"{b}Loaded {nb_all_runs} runs from {WANDB_CSV.name}{e}.")


def print_runs_info(df: pd.DataFrame, columns: list) -> None:
    """Print run_name and selected columns for each row."""
    for run_id, row in df.iterrows():
        print(f"  - {run_id:<12} {row['run_name']:<50}:", end="")
        for col in columns:
            print(f" {r}{col}={row[col]}{e},", end="")
        print()


def clean_runs(df: pd.DataFrame) -> pd.DataFrame:
    """Validity filters shared by every figure: dataset, seeds, drop broken runs.
    NOTE: does NOT select lr or activation — callers do that explicitly."""
    out = df[df["data_name"].isin(["TSXSSCDataModule"])].copy()
    out = out[out["seed"].isin([0, 1, 2, 3, 4, 5, 6])]
    # `debug`/`crashed` are validity tags (broken runs), not an lr selector.
    for tag in ["debug", "crashed"]:
        out = out[~out["tags"].apply(lambda ts: tag in ts)]
    out = out[~out["model_name"].str.contains("gdn1", na=False)]
    return out


def select_prod_lr(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only each architecture's production lr (by VALUE, the `lr` column)."""
    keep = df.apply(
        lambda row: (
            row["architecture"] in PROD_LR
            and abs(float(row["lr"]) - PROD_LR[row["architecture"]]) < 1e-12
        ),
        axis=1,
    )
    return df[keep]


# ── Production dataset (F2 / F3): ReLU + no_output_padding + per-arch PROD_LR ──
runs_df = clean_runs(raw_runs_df)
runs_df = runs_df[runs_df["model_name"].str.contains("_relu", case=False, na=False)]
runs_df = runs_df[runs_df["no_output_padding"] == True]
runs_df = select_prod_lr(runs_df)
# one row per (arch, λ, seed) — keep the latest if a run was re-evaluated
runs_df = runs_df.sort_index().drop_duplicates(["architecture", "lmbda", "seed"], keep="last")

nb_kept = len(runs_df)
print(
    f"\n{g}Production set:{e} {nb_kept} runs "
    f"(per-arch PROD_LR={ {a: f'{lr:.0e}' for a, lr in PROD_LR.items()} }).\n"
    f"Runs per architecture (expect 60 each):\n{runs_df['architecture'].value_counts().to_string()}"
)
if VERBOSE_FILTERING:
    print("\nlr actually selected per architecture:")
    print(runs_df.groupby("architecture")["lr"].unique().to_string())

unique_lmbda = sorted(runs_df["lmbda"].dropna().unique().astype(float).tolist())
unique_seeds = sorted(runs_df["seed"].dropna().unique().astype(int).tolist())
unique_archs = runs_df["architecture"].unique().tolist()
print(
    f"\nUnique architectures ({g}{len(unique_archs)}{e}): {unique_archs}"
    f"\nUnique λ ({g}{len(unique_lmbda)}{e}): {unique_lmbda}"
    f"\nUnique seeds ({g}{len(unique_seeds)}{e}): {unique_seeds}"
)

## Build Statistics

In [ ]:
# ── Metrics tracked per (architecture, activation, output_padding) group ──
# Reference = the 500-image MERLIN subset (METRIC_PREFIX) + the rANS bitstream bpp.
METRICS_OF_INTEREST = [f"{METRIC_PREFIX}/{BPP_METRIC}"]
for metric in ["psnr", "ssim", "ms_ssim", "epd"]:
    METRICS_OF_INTEREST.append(f"{METRIC_PREFIX}/{metric}_noisy")
    METRICS_OF_INTEREST.append(f"{METRIC_PREFIX}/{metric}_merlin")
    METRICS_OF_INTEREST.append(f"{METRIC_PREFIX}/{metric}_adam_noc")
STATS = ["mean", "min", "max", "std"]
STAT_COLS = [f"{m} {s}" for m in METRICS_OF_INTEREST for s in STATS]
print(
    f"Tracking {len(METRICS_OF_INTEREST)} metrics x {len(STATS)} stats = {len(STAT_COLS)} columns"
    f"  (prefix={METRIC_PREFIX}, bpp={BPP_METRIC})"
)


def build_statistics(df: pd.DataFrame, verbose: bool = True) -> dict:
    """Group runs by (architecture, model_name, no_output_padding) and aggregate
    METRICS_OF_INTEREST over seeds per λ. Key: '{arch}_{activation}[_out_pad]'.
    Group on `architecture` (not just model_name) because ResSHyp and SHyp share
    the same PyTorch class name and would otherwise be merged."""
    stats: dict = {}
    for (arch, model_name, no_out_pad), df_group in df.groupby(
        ["architecture", "model_name", "no_output_padding"]
    ):
        activation = model_name.split("_")[-1]  # "relu" or "gdn"
        key = f"{arch}_{activation}{'' if no_out_pad else '_out_pad'}"
        if verbose:
            print(f"  {key:<30}  n={len(df_group)}")

        lmbda_values = sorted(df_group["lmbda"].unique().astype(float).tolist())
        stats_df = pd.DataFrame(index=lmbda_values, columns=STAT_COLS, dtype=float)
        for lmbda, df_lmbda in df_group.groupby("lmbda"):
            for metric in METRICS_OF_INTEREST:
                if metric not in df_lmbda.columns:
                    continue  # metric absent from CSV (older eval code)
                numeric = pd.to_numeric(df_lmbda[metric], errors="coerce")
                for stat in STATS:
                    stats_df.loc[lmbda, f"{metric} {stat}"] = getattr(numeric, stat)()
        stats[key] = stats_df
    return stats


# Production statistics (per-arch PROD_LR, ReLU + no_output_padding) → F2/F3.
model_statistics = build_statistics(runs_df)
print(f"\n{b}model_statistics keys:{e} {list(model_statistics.keys())}")

### NaN / Missing-Run Diagnostic

For each group where a statistic is NaN, list the affected runs with their IDs and
which metrics are absent from the W&B summary.

Metrics can be missing in two ways:
- **(a) key absent or `None`** in the W&B summary
- **(b) key present but stored as `float NaN`** — most common cause of NaN `std` when `mean` looks valid
  (5/6 seeds have NaN → pandas `.mean()` skips them, but `.std(ddof=1)` needs ≥ 2 values)

In [ ]:
NAN_CHECK_METRICS = [
    f"{METRIC_PREFIX}/{BPP_METRIC}",
    f"{METRIC_PREFIX}/psnr_merlin",
    f"{METRIC_PREFIX}/ssim_merlin",
    f"{METRIC_PREFIX}/epd_merlin",
]
CHECK_STAT_COLS = [
    f"{m} {s}"
    for m in [f"{METRIC_PREFIX}/{BPP_METRIC}", f"{METRIC_PREFIX}/psnr_merlin"]
    for s in ["mean", "std"]
]


def nan_diagnostic(statistics: dict, source_df: pd.DataFrame) -> list:
    """List runs behind any NaN statistic (missing/NaN W&B summary metric)."""
    nan_run_ids: list = []
    print(f"{y}Runs with NaN / missing metrics:{e}\n")
    print(f"  {'Run ID':<12} {'Name':<55} {'λ':>6}  {'seed':>4}  {'no_out_pad':>10}  Status")
    print(f"  {'-' * 12} {'-' * 55} {'-' * 6}  {'-' * 4}  {'-' * 10}  ------")
    for model_key, stats_df in statistics.items():
        nan_lmbdas = set()
        for col in CHECK_STAT_COLS:
            if col in stats_df.columns:
                nan_lmbdas.update(stats_df.index[stats_df[col].isna()].astype(float).tolist())
        if not nan_lmbdas:
            continue
        has_out_pad = model_key.endswith("_out_pad")
        base = model_key[: -len("_out_pad")] if has_out_pad else model_key
        arch, activation = base.split("_")[0], base.split("_")[1]
        no_out_pad = not has_out_pad
        candidate_runs = source_df[
            (source_df["architecture"] == arch)
            & (source_df["model_name"].str.endswith(f"_{activation}"))
            & (source_df["no_output_padding"] == no_out_pad)
            & (source_df["lmbda"].isin(nan_lmbdas))
        ]
        for run_id, run_row in candidate_runs.iterrows():
            missing = [
                m for m in NAN_CHECK_METRICS if m not in run_row.index or pd.isna(run_row[m])
            ]
            nan_run_ids.append(run_id)
            flag = f"{r}MISSING/NaN{e}: {missing}" if missing else f"{g}present{e}"
            print(
                f"  {run_id:<12} {run_row['run_name']:<55} {run_row['lmbda']:>6.0f}"
                f"  {str(int(run_row['seed'])):>4}  {str(no_out_pad):>10}  {flag}"
            )
    nan_run_ids = list(dict.fromkeys(nan_run_ids))
    print(f"\n{y}→ {len(nan_run_ids)} runs with missing stats.{e}")
    return nan_run_ids


nan_run_ids = nan_diagnostic(model_statistics, runs_df)

## Plot Helpers

All plots share two composable functions:
- `plot_rd_curves(ax, …)` — draws curves on an existing axes (no labels / legend)
- `make_rd_figure(…)` — creates a standalone figure
- `make_rd_grid(…)` — creates a multi-subplot grid

Visual encoding is injected via callables so each section defines its own mapping:
- `color_fn(key)` / `linestyle_fn(key)` / `label_fn(key)` / `legend_fn(ax)` / `key_filter(key)`

In [ ]:
def plot_rd_curves(
    ax,
    statistics: dict,
    ref: str,
    color_fn,
    linestyle_fn,
    marker_fn=None,
    markersize=4,
    prefix: str = METRIC_PREFIX,
    key_filter=None,
    label_fn=None,
    mean_std_error_bands: bool = True,
    alpha_error_bars: float = 0.1,
    annotate_key=None,
    annotate_lambdas=None,
) -> None:
    """Draw one RD curve per key in *statistics* on *ax*. No axis labels or legend.
    x = {prefix}/{BPP_METRIC} (rANS bitstream bpp); y = {prefix}/{ref}.

    annotate_key:    key whose data points are labelled with their λ value.
    annotate_lambdas: None = label all points; list = label only those λ values.
    """
    for key, stats_df in statistics.items():
        if key_filter is not None and not key_filter(key):
            continue

        bpp_col = f"{prefix}/{BPP_METRIC} mean"
        bpp_std_col = f"{prefix}/{BPP_METRIC} std"
        q_col = f"{prefix}/{ref} mean"
        q_min_col = f"{prefix}/{ref} min"
        q_max_col = f"{prefix}/{ref} max"
        q_mean_col = f"{prefix}/{ref} mean"
        q_std_col = f"{prefix}/{ref} std"

        if bpp_col not in stats_df.columns or q_col not in stats_df.columns:
            continue

        bpp = stats_df[bpp_col].astype(float).to_numpy()
        q = stats_df[q_col].astype(float).to_numpy()
        valid = ~(np.isnan(bpp) | np.isnan(q))
        if valid.sum() == 0:
            continue

        idx = np.argsort(bpp[valid])
        bpp_s, q_s = bpp[valid][idx], q[valid][idx]
        color = color_fn(key)
        linestyle = linestyle_fn(key)
        marker = marker_fn(key) if marker_fn is not None else "o"
        label = label_fn(key) if label_fn else None

        ax.plot(
            bpp_s,
            q_s,
            marker=marker,
            linestyle=linestyle,
            color=color,
            markersize=markersize,
            label=label,
        )

        # BPP error bars (std across seeds)
        if bpp_std_col in stats_df.columns:
            bpp_std = stats_df[bpp_std_col].astype(float).to_numpy()[valid][idx]
            if not np.all(np.isnan(bpp_std)):
                ax.errorbar(
                    bpp_s,
                    q_s,
                    xerr=bpp_std,
                    fmt="none",
                    ecolor=PALETTE["metrics"]["error_bars"],
                    capsize=2,
                    alpha=0.5,
                )

        # Mean ± std (or min/max) shaded band across seeds
        if mean_std_error_bands:
            if q_mean_col in stats_df.columns and q_std_col in stats_df.columns:
                q_mean = stats_df[q_mean_col].astype(float).to_numpy()[valid][idx]
                q_std = stats_df[q_std_col].astype(float).to_numpy()[valid][idx]
                if not np.all(np.isnan(q_mean)):
                    ax.fill_between(
                        bpp_s, q_mean - q_std, q_mean + q_std, alpha=alpha_error_bars, color=color
                    )
        else:
            if q_min_col in stats_df.columns and q_max_col in stats_df.columns:
                q_min = stats_df[q_min_col].astype(float).to_numpy()[valid][idx]
                q_max = stats_df[q_max_col].astype(float).to_numpy()[valid][idx]
                if not np.all(np.isnan(q_min)):
                    ax.fill_between(bpp_s, q_min, q_max, alpha=alpha_error_bars, color=color)

        # λ annotations: quarter-circle placement — i=0 (low bpp) → left, i=N-1 → above.
        if annotate_key is not None and key == annotate_key:
            lambdas_s = np.array(stats_df.index[valid].tolist(), dtype=float)[idx]
            n_pts = len(lambdas_s)
            for i, (lam, xp, yp) in enumerate(zip(lambdas_s, bpp_s, q_s)):
                if annotate_lambdas is not None and lam not in annotate_lambdas:
                    continue
                # angle sweeps 180° (left) → 90° (up) as i goes 0 → N-1
                t = i / max(n_pts - 1, 1)
                angle_rad = np.radians(180.0 - 90.0 * t)
                dx = 9.0 * np.cos(angle_rad)
                dy = 9.0 * np.sin(angle_rad)
                ax.annotate(
                    f"λ={int(lam)}",
                    (xp, yp),
                    textcoords="offset points",
                    xytext=(dx, dy),
                    fontsize=8.5,
                    color=color,
                    alpha=0.85,
                    ha="right" if dx < -1.0 else "center",
                    va="center" if abs(dy) < 4.0 else "bottom",
                )


def make_rd_figure(
    statistics: dict,
    ref: str,
    ylabel: str,
    color_fn,
    linestyle_fn,
    marker_fn=None,
    markersize=4,
    key_filter=None,
    label_fn=None,
    legend_fn=None,
    title: str = "",
    save_name: str = None,
    manuscript_name: str = None,
    figsize: tuple = (9, 6),
    mean_std_error_bands: bool = True,
    alpha_error_bars: float = 0.1,
    legend_loc: str = "best",
    annotate_arch=None,
    annotate_lambdas=None,
) -> None:
    """Create a standalone figure with one set of RD curves.
    save_name → results/plots/RD-curves/; manuscript_name → figures/images/ (paper).

    legend_loc:      matplotlib loc string (default "best"); ignored when legend_fn is set.
    annotate_arch:   key in statistics whose points are labelled with their λ value.
    annotate_lambdas: None = label all points; list = label only those λ values.
    """
    fig, ax = plt.subplots(figsize=figsize)
    plot_rd_curves(
        ax,
        statistics,
        ref,
        color_fn,
        linestyle_fn,
        marker_fn=marker_fn,
        markersize=markersize,
        key_filter=key_filter,
        label_fn=label_fn,
        mean_std_error_bands=mean_std_error_bands,
        alpha_error_bars=alpha_error_bars,
        annotate_key=annotate_arch,
        annotate_lambdas=annotate_lambdas,
    )
    ax.set_xlabel("Bitrate [bpp]", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    if title:
        ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if "ssim" in ref:
        ax.set_ylim(0.6, 1.0)
    if legend_fn:
        legend_fn(ax)
    elif label_fn:
        ax.legend(fontsize=12, loc=legend_loc)
    plt.tight_layout()
    if save_name and SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"{save_name}.{FIGURE_FORMAT}", bbox_inches="tight")
        export_manuscript(fig, manuscript_name)
    plt.show()


def make_rd_grid(
    statistics: dict,
    metrics: list,
    color_fn,
    linestyle_fn,
    marker_fn=None,
    key_filter=None,
    label_fn=None,
    legend_fn=None,
    title: str = "",
    save_name: str = None,
    ncols: int = 3,
    alpha_error_bars: float = 0.1,
    mean_std_error_bands: bool = True,
    xlabel: str = "Bitrate [bpp]",
) -> None:
    """Create a grid of RD-curve subplots, one per metric."""
    n = len(metrics)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 5 * nrows), squeeze=False)

    for ax, (ref, ylabel) in zip(axes.flat, metrics):
        plot_rd_curves(
            ax,
            statistics,
            ref,
            color_fn,
            linestyle_fn,
            marker_fn=marker_fn,
            key_filter=key_filter,
            label_fn=label_fn,
            alpha_error_bars=alpha_error_bars,
            mean_std_error_bands=mean_std_error_bands,
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel)
        ax.grid(True, alpha=0.3)
        if "epd" in ref:
            ax.set_ylim(0, 1.2)
        if "ssim" in ref:
            ax.set_ylim(0.6, 1.0)
        if legend_fn:
            legend_fn(ax)
        elif label_fn:
            ax.legend(fontsize=10)

    for ax in list(axes.flat)[n:]:
        ax.set_visible(False)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    if save_name and SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"{save_name}.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()

## Ablation: ResSHyp Variants

Four configurations of ResSHyp: activation (ReLU / GDN) × output_padding (with / without).

- **Color** → activation (ReLU = green, GDN = mauve)
- **Linestyle** → output padding (solid = without, dotted = with)
- **Band** → min/max range across 6 seeds  |  **x-bars** → BPP std

In [ ]:
# ── Ablation dataset: ResSHyp @ lr=5e-5, ReLU + GDN, output_padding on/off ──
# Kept at the OLD default lr (5e-5): the GDN variants only exist there, so this is a
# controlled A/B at one lr. (The production figures F2/F3 use ResSH @ 1e-4.)
ablation_runs_df = clean_runs(raw_runs_df)
ablation_runs_df = ablation_runs_df[
    (ablation_runs_df["architecture"] == "ResSHyp")
    & (ablation_runs_df["lr"].round(8) == round(ABLATION_LR, 8))
]
ablation_runs_df = ablation_runs_df.sort_index().drop_duplicates(
    ["architecture", "model_name", "no_output_padding", "lmbda", "seed"], keep="last"
)
print(f"{b}Ablation set (ResSHyp @ lr={ABLATION_LR:.0e}):{e} {len(ablation_runs_df)} runs")
ablation_statistics = build_statistics(ablation_runs_df)

# ── GDN vs ReLU @ λ=1000 ─────────────────────────────────────────────────────
# Raw CSV uses "lmbda" and "test_sub500/metric" column names.
_CAL_KW = dict(lambda_col="lmbda", metric_prefix=f"{METRIC_PREFIX}/", bpp_col=BPP_METRIC)
compare_at_lambda(
    ablation_runs_df[
        ablation_runs_df["model_name"].str.endswith("_relu")
        & (ablation_runs_df["no_output_padding"] == True)
    ],
    ablation_runs_df[
        ablation_runs_df["model_name"].str.endswith("_gdn")
        & (ablation_runs_df["no_output_padding"] == True)
    ],
    "ReLU",
    "GDN",
    lmbda=1000.0,
    **_CAL_KW,
)


# ── Visual encoding for ablation ──────────────────────────────────────────────
# key format: "{arch}_{activation}[_out_pad]"  e.g. "ResSHyp_relu_out_pad"
def ablation_color(key: str) -> str:
    # All ablation curves share the ResSH architecture colour; activation is encoded
    # by marker (ablation_marker), output_padding by linestyle. Keeps the figure off the
    # green/pink architecture palette used by the 4-arch and cross-precision figures.
    return PALETTE["architectures"]["ResSHyp"]


def ablation_linestyle(key: str) -> str:
    return "dotted" if key.endswith("_out_pad") else "solid"


def ablation_marker(key: str) -> str:
    # ReLU -> circle, GDN -> cross.
    return "o" if key.split("_")[1] == "relu" else "x"


def ablation_legend(ax, fontsize: int = 12) -> None:
    # All curves share the ResSH colour, so the legend uses neutral black symbols to denote
    # the encoding dimensions: marker = activation, linestyle = output_padding.
    handles = [
        Line2D([0], [0], color="black", marker="o", markersize=6, linestyle="none", label="ReLU"),
        Line2D([0], [0], color="black", marker="x", markersize=6, linestyle="none", label="GDN"),
        Line2D([0], [0], color="black", linestyle="solid", label=r"without $op$"),
        Line2D([0], [0], color="black", linestyle="dotted", label=r"with $op$"),
    ]
    ax.legend(handles=handles, fontsize=fontsize)


def ABLATION_FILTER(key):
    return key.split("_")[0] == "ResSHyp"


ABLATION_METRICS = [
    ("psnr_merlin", "PSNR [dB]"),
    ("ssim_merlin", "SSIM"),
    # ("epd_merlin", "EPD"),
]

# ── F1 — single PSNR figure (→ paper: fig_ablation_RD) ────────────────────────
make_rd_figure(
    ablation_statistics,
    ref="psnr_merlin",
    ylabel="PSNR [dB]",
    color_fn=ablation_color,
    linestyle_fn=ablation_linestyle,
    marker_fn=ablation_marker,
    markersize=6,
    key_filter=ABLATION_FILTER,
    alpha_error_bars=0.2,
    legend_fn=ablation_legend,
    save_name="RD-curves_ablation_PSNR-MERLIN",
    manuscript_name="fig_ablation_RD",
)

In [ ]:
# ── 3-metric grid (PSNR / SSIM / EPD) ────────────────────────────────────────
make_rd_grid(
    ablation_statistics,
    metrics=ABLATION_METRICS,
    color_fn=ablation_color,
    linestyle_fn=ablation_linestyle,
    marker_fn=ablation_marker,
    key_filter=ABLATION_FILTER,
    legend_fn=ablation_legend,
    title=f"ResSHyp ablation @ lr={ABLATION_LR:.0e}  (MERLIN ref)",
    save_name="RD-curves_ablation_subplots",
)

## Architecture Comparison

All four architectures (**ResSHyp**, **SHyp**, **ResFP**, **FP**) on the ReLU + no output_padding baseline.

- **Color** → architecture (Okabe-Ito palette)
- **Band** → min/max range across seeds  |  **x-bars** → BPP std

In [ ]:
# ── Derive arch_statistics from model_statistics ─────────────────────────────
# The common baseline for all architectures is ReLU + no_output_padding.
# "ResSHyp_relu" (no _out_pad suffix) is exactly that entry in model_statistics.
ARCH_ORDER = ["ResSHyp", "SHyp", "ResFP", "FP"]
ARCH_COLORS = {
    arch: PALETTE["architectures"][arch] for arch in ARCH_ORDER if arch in PALETTE["architectures"]
}
ARCH_DISPLAY = {"ResSHyp": "ResSH", "SHyp": "SH", "ResFP": "ResFP", "FP": "FP"}

arch_statistics = {
    arch: model_statistics[f"{arch}_relu"]
    for arch in ARCH_ORDER
    if f"{arch}_relu" in model_statistics
}
print(f"Architectures in baseline: {list(arch_statistics.keys())}")


# ── Visual encoding ───────────────────────────────────────────────────────────
def arch_color(key: str) -> str:
    return ARCH_COLORS.get(key, "#999999")


def arch_linestyle(key: str) -> str:
    return "solid"


def arch_label(key: str) -> str:
    return ARCH_DISPLAY.get(key, key)

In [ ]:
# ── Architecture pairwise comparisons at fixed λ ─────────────────────────────
# runs_df is filtered to ReLU + no_output_padding + production lr.
# Raw CSV uses "lmbda" and "test_sub500/metric" column names.
_CAL_KW = dict(lambda_col="lmbda", metric_prefix=f"{METRIC_PREFIX}/", bpp_col=BPP_METRIC)

compare_at_lambda(
    runs_df[runs_df["architecture"] == "ResSHyp"],
    runs_df[runs_df["architecture"] == "SHyp"],
    "ResSH",
    "SH",
    lmbda=1000.0,
    **_CAL_KW,
)

compare_at_lambda(
    runs_df[runs_df["architecture"] == "ResSHyp"],
    runs_df[runs_df["architecture"] == "SHyp"],
    "ResSH",
    "SH",
    lmbda=500.0,
    **_CAL_KW,
)

compare_at_lambda(
    runs_df[runs_df["architecture"] == "ResSHyp"],
    runs_df[runs_df["architecture"] == "SHyp"],
    "ResSH",
    "SH",
    lmbda=200.0,
    **_CAL_KW,
)

compare_at_lambda(
    runs_df[runs_df["architecture"] == "ResFP"],
    runs_df[runs_df["architecture"] == "FP"],
    "ResFP",
    "FP",
    lmbda=1000.0,
    **_CAL_KW,
)

compare_at_lambda(
    runs_df[runs_df["architecture"] == "ResSHyp"],
    runs_df[runs_df["architecture"] == "ResFP"],
    "ResSH",
    "ResFP",
    lmbda=20.0,
    **_CAL_KW,
)

In [ ]:
# ── F2 — one figure per metric (PSNR → paper: fig_4arch_RD) ───────────────────
ARCH_METRICS = [
    ("psnr_merlin", "PSNR [dB]"),
    ("ssim_merlin", "SSIM"),
    ("epd_merlin", "EPD"),
]
# Only the PSNR panel is the paper figure; SSIM/EPD are kept for inspection.
MANUSCRIPT_BY_METRIC = {"psnr_merlin": "fig_4arch_RD"}

# ── Annotation / legend config ─────────────────────────────────────────────────
LEGEND_LOC = "lower right"
ANNOTATE_ARCH = "ResSHyp"  # key in arch_statistics to annotate with λ labels
ANNOTATE_LAMBDAS = None  # None = all 10 points; e.g. [1, 5, 20, 100, 500, 1000]
# ──────────────────────────────────────────────────────────────────────────────

for ref, ylabel in ARCH_METRICS:
    make_rd_figure(
        arch_statistics,
        ref=ref,
        ylabel=ylabel,
        color_fn=arch_color,
        linestyle_fn=arch_linestyle,
        label_fn=arch_label,
        # title=f"Architecture comparison — {ylabel} (ReLU, no output_padding, production lr)",
        save_name=f"RD-curves_arch_comparison_{ref.replace('/', '_')}",
        manuscript_name=MANUSCRIPT_BY_METRIC.get(ref),
        alpha_error_bars=0.1,
        legend_loc=LEGEND_LOC,
        annotate_arch=ANNOTATE_ARCH,
        annotate_lambdas=ANNOTATE_LAMBDAS,
    )

## Quality vs Deployment Cost (F3 — numbers)

Per-architecture **PSNR at a fixed iso-rate (1.0 bpp)** against **single-patch FPGA latency** and
architectural **#OPs**. Used as numbers in the prose (not a paper figure); the Pareto scatter below
is exploratory (`SHOW_PARETO`).

In [ ]:
# ── F3 — quality vs deployment cost (NUMBERS for the prose; plot is optional) ──
# Quality = PSNR at a fixed iso-rate (interp on the production RD curve).
# Cost  = single-patch FPGA latency (s0) + architectural #OPs (lr-independent).
# B2: iso-rate default 0.25 bpp (≈ RD knee); exact-bpp column; LaTeX tabular output.
TARGET_BPP = 0.25  # change this to explore other iso-rates (e.g. 1.0)
BENCH_DIR = ROOT_DIR / "results" / "benchmark_hardware"


def arch_rd_arrays(arch: str, ref: str = "psnr_merlin"):
    """(bpp, quality) mean arrays for one arch, sorted by bpp, from arch_statistics."""
    s = arch_statistics[arch]
    bpp = s[f"{METRIC_PREFIX}/{BPP_METRIC} mean"].astype(float).to_numpy()
    q = s[f"{METRIC_PREFIX}/{ref} mean"].astype(float).to_numpy()
    m = ~(np.isnan(bpp) | np.isnan(q))
    o = np.argsort(bpp[m])
    return bpp[m][o], q[m][o]


def psnr_at_bpp(arch: str, target: float = TARGET_BPP) -> float:
    bpp, q = arch_rd_arrays(arch)
    return float(np.interp(target, bpp, q))


def exact_bpp_psnr(arch: str, target: float = TARGET_BPP):
    """(bpp, psnr) at the λ whose mean bpp is closest to target (no interpolation)."""
    bpp, q = arch_rd_arrays(arch)
    if len(bpp) == 0:
        return float("nan"), float("nan")
    idx = int(np.argmin(np.abs(bpp - target)))
    return float(bpp[idx]), float(q[idx])


def fpga_cost(arch: str):
    """(#OPs[G], latency_full_ms, latency_compress_ms) for the s0 reference run."""
    model = f"{arch}-relu_s0_L1000_pt"
    xi = json.load(open(BENCH_DIR / "_roofline" / f"{model}_xmodel_info.json"))
    # xdputil reports `workload` in MACs; Xilinx counts 1 MAC = 2 ops (docs/FPGA_benchmark.md),
    # so double it to match the GOPs convention used in tab:network_stats.
    ops = 2 * xi["total_workload_ops"] / 1e9
    lat_full = lat_comp = float("nan")
    for f in (BENCH_DIR / model).glob("*.json"):
        d = json.load(open(f))
        if d.get("config") == "s0" and d.get("scenario") == "full":
            lat_full = d["total_latency_mean_ms"]
        if d.get("config") == "s0" and d.get("scenario") == "compress":
            lat_comp = d["total_latency_mean_ms"]
    return ops, lat_full, lat_comp


# ── Build table rows ────────────────────────────────────────────────────────────
f3_rows = {}
for arch in ARCH_ORDER:
    if arch not in arch_statistics:
        continue
    ops, lat_full, lat_comp = fpga_cost(arch)
    bpp_arr, q_arr = arch_rd_arrays(arch)
    exact_b, exact_q = exact_bpp_psnr(arch, TARGET_BPP)
    f3_rows[arch] = dict(
        ops=ops,
        lat_full=lat_full,
        lat_comp=lat_comp,
        psnr_iso=psnr_at_bpp(arch, TARGET_BPP),
        psnr_max=float(q_arr[-1]) if len(q_arr) else float("nan"),
        exact_bpp=exact_b,
        exact_psnr=exact_q,
    )

# ── Clean text table ────────────────────────────────────────────────────────────
hdr = (
    f"  {'Arch':6}  {'#OPs[G]':>8}  {'lat_full':>10}  {'lat_comp':>10}  "
    f"{'PSNR@{:.2f}(interp)'.format(TARGET_BPP):>22}  {'exact_bpp':>10}  "
    f"{'PSNR@exact':>11}  {'PSNR@maxλ':>10}"
)
print(f"\n{b}F3 — quality vs cost  (iso-rate @ {TARGET_BPP} bpp){e}")
print(hdr)
print("  " + "-" * len(hdr.rstrip()))
for arch in ARCH_ORDER:
    if arch not in f3_rows:
        continue
    rr = f3_rows[arch]
    print(
        f"  {ARCH_DISPLAY.get(arch, arch):6}  {rr['ops']:8.2f}  {rr['lat_full']:10.1f}  "
        f"{rr['lat_comp']:10.1f}  {rr['psnr_iso']:22.2f}  {rr['exact_bpp']:10.3f}  "
        f"{rr['exact_psnr']:11.2f}  {rr['psnr_max']:10.2f}"
    )

# ── LaTeX tabular ──────────────────────────────────────────────────────────────
print(f"\n{b}LaTeX tabular:{e}")
_latex = [
    r"\begin{tabular}{lrrrrrr}",
    r"\toprule",
    (
        r"Arch & GOPs & Lat$_\mathrm{{full}}$ [ms] & Lat$_\mathrm{{comp}}$ [ms] & "
        r"PSNR @{:.2f}\,bpp [dB] & Exact bpp & PSNR @exact [dB] \\".format(TARGET_BPP)
    ),
    r"\midrule",
]
for arch in ARCH_ORDER:
    if arch not in f3_rows:
        continue
    rr = f3_rows[arch]
    _latex.append(
        f"{ARCH_DISPLAY.get(arch, arch)} & {rr['ops']:.1f} & {rr['lat_full']:.1f} & "
        f"{rr['lat_comp']:.1f} & {rr['psnr_iso']:.2f} & {rr['exact_bpp']:.3f} & "
        r"{:.2f} \\".format(rr["exact_psnr"])
    )
_latex += [r"\bottomrule", r"\end{tabular}"]
print("\n".join(_latex))

# ── Optional Pareto scatter (exploratory; not a paper figure) ─────────────────
SHOW_PARETO = True
if SHOW_PARETO:
    fig, ax = plt.subplots(figsize=(7, 5))
    for arch in ARCH_ORDER:
        if arch not in f3_rows:
            continue
        rr = f3_rows[arch]
        ax.scatter(
            rr["lat_full"],
            rr["psnr_iso"],
            s=120,
            color=arch_color(arch),
            edgecolor="k",
            linewidth=0.6,
            zorder=3,
        )
        ax.annotate(
            f"{ARCH_DISPLAY.get(arch, arch)}\n({rr['ops']:.0f} GOPs)",
            (rr["lat_full"], rr["psnr_iso"]),
            textcoords="offset points",
            xytext=(8, 4),
            fontsize=9,
            color=arch_color(arch),
            fontweight="bold",
        )
    ax.set_xlabel("FPGA latency, full inference [ms]  (s0, single patch)")
    ax.set_ylabel(f"PSNR @ {TARGET_BPP} bpp [dB]  (iso-rate)")
    ax.set_title("F3 — quality vs deployment cost (exploratory)")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"RD-curves_pareto.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()